# WildSpeech M8.4 — Phase 0: Trusted Evaluation & Reference Parity

**MISSION:** freeze the official-data evaluation, reproduce DrivenData's 14-feature RidgeCV reference (R²≈0.193 mean of 50 folds), establish the run ledger, and test the historical M8.3 models on exactly matching folds **without inventing results**.

**No audio extraction or GPU work is needed.** This notebook mounts Google Drive first, clones the GitHub package, validates official CSVs, creates 50 reference RepeatedKFold splits and 50 school-disjoint robustness splits, runs all 100 reference fits, verifies held-out predictions, and produces a rigorously checked submission CSV under a private Drive output directory. The notebook does **not** submit to the leaderboard.

Research protocol: primary metric = 10-repeat mean pooled OOF R²; published-reference parity metric = mean of 50 fold R² values. Every future comparison must share the frozen fold hash and be preregistered before model fitting. No training on test rows, no test-derived fitted transforms, and no test pseudo-labeling.

Reference: https://blog.drivendata.org/blog/literacy-speech-modeling-reference

## 0. Mount Drive **before** touching competition data

In Colab click Run all, authorize Drive, and confirm the dataset is in MyDrive/WildSpeech. Locally, set WILDSPEECH_DRIVE_ROOT to your mounted folder.

In [ ]:
import os, sys, json, subprocess, importlib, warnings, time
from pathlib import Path
from IPython.display import display, Markdown
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE=Path("/content/drive/MyDrive")
else:
    DRIVE=Path(os.environ.get("WILDSPEECH_DRIVE_ROOT",str(Path.home()/"MyDrive"))).expanduser()
assert DRIVE.is_dir(), f"Drive not mounted at {DRIVE}"
print("✓ Google Drive:",DRIVE)
print("WildSpeech folders:",[p.name for p in DRIVE.iterdir() if p.is_dir() and "wild" in p.name.lower()])

## 1. Install the exact GitHub source

Installs the local M8/M8.4 package. No ZIP upload. A dirty existing checkout cannot be force-reset; only a fast-forward update is allowed.

In [ ]:
URL="https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT=Path("/content/WildSpeech_git") if IN_COLAB else Path.cwd()/"WildSpeech_git"
PKG=CHECKOUT/"WildSpeech_M8"
if not (CHECKOUT/".git").is_dir():
    if CHECKOUT.exists(): raise RuntimeError(f"Non-Git checkout exists at {CHECKOUT}")
    subprocess.run(["git","clone","--depth","1","--branch","main",URL,str(CHECKOUT)],check=True)
else:
    subprocess.run(["git","-C",str(CHECKOUT),"pull","--ff-only","origin","main"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PKG)],check=True)
SOURCE=str(PKG/"src")
if SOURCE not in sys.path:sys.path.insert(0,SOURCE)
GIT_SHA=subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
from wildspeech_m84.data import locate_data_root,load_official
from wildspeech_m84.folds import build_folds,freeze_folds,validate_folds
from wildspeech_m84.reference import reference_frames,REFERENCE_COLUMNS
from wildspeech_m84.runner import run_phase0
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("✓ Package installed at commit:",GIT_SHA)

## 2. One configuration cell: official files and private output paths

Leave overrides blank when your Drive uses the existing WildSpeech layout. Historial M8.3 research can optionally run, but **cannot be promoted** until its old feature-generation test-dependence is audited. M7.3's original implementation is absent from repository main.

In [ ]:
DATA_ROOT_OVERRIDE=""                    # autodiscover six official CSVs
OUTPUT_ROOT_OVERRIDE=""                  # defaults to MyDrive/WildSpeech_M84/phase0
M8_ROOT_OVERRIDE=""                      # defaults to MyDrive/WildSpeech_M8
M7_PROFILE_OVERRIDE=""                   # optional prior M7 profile path
RUN_LEGACY_RESEARCH_ONLY=False           # separate research check; default OFF
REQUIRE_REFERENCE_PARITY=True            # hard stop if mean fold R² outside [0.188,0.198]
ROOT=Path(OUTPUT_ROOT_OVERRIDE).expanduser() if OUTPUT_ROOT_OVERRIDE else DRIVE/"WildSpeech_M84"/"phase0"
ROOT.mkdir(parents=True,exist_ok=True)
print("Phase 0 private output root:",ROOT)
print("Reference acceptance range:",(0.188,0.198))

## 3. Data and reference feature preflight — no training

Check six official CSVs, student and label alignment, reference-compatible metadata row order, and the 14 exact reference columns. Train and test student IDs must be disjoint.

In [ ]:
DATA_ROOT=locate_data_root(DRIVE,DATA_ROOT_OVERRIDE)
D=load_official(DATA_ROOT,ROOT)
X_REF,X_TEST_REF,Y=reference_frames(D)
assert X_REF.shape==(len(Y),14) and X_TEST_REF.shape==(len(D.test_students),14)
assert X_REF.index.tolist()==D.train_students.index.tolist()
assert X_TEST_REF.index.tolist()==D.submission_format.index.tolist()
print("Train/test:",X_REF.shape,X_TEST_REF.shape,
      "| Train clips:",len(D.train_clips),"| Test clips:",len(D.test_clips))
print("Reference columns:",list(X_REF.columns))
print("Labels order equal to metadata order:",D.order_manifest["labels_original_order_equal"])
print("Canonical student order hash:",D.order_manifest["canonical_student_order_sha256"][:18])
display(X_REF.notna().mean().rename("train_feature_coverage").to_frame().T)
print("Reference data read WITHOUT fitting on test students.")

## 4. Freeze 100 folds and verify the split contract

**Student CV:** 5 folds × 10 repeats of RepeatedKFold(seed=42), ordered identically to the reference's metadata frame.

**School robustness:** 5 school-disjoint folds × 10 seeded repetitions, with balanced group assignments. School IDs are split labels, not model features.

In [ ]:
FOLDS=build_folds(D.ids,D.groups)
assert validate_folds(FOLDS,D.ids,D.groups)
FOLD_HASH=freeze_folds(FOLDS,D.ids,D.groups,ROOT)
display(FOLDS.groupby("scheme")[["n_train","n_valid"]].agg(["min","mean","max"]).round(1))
print("Frozen fold hash:",FOLD_HASH)
print("Fold file:",ROOT/"folds"/"folds.parquet")
print("Manifest:",ROOT/"manifests"/"fold_manifest.json")

## 5. Read and freeze the preregistered experiment plan

This is recorded **before** model training, in an append-only ledger. Historical models that cannot be faithfully reproduced remain NOT_REPRODUCED; the notebook will not invent past scores.

In [ ]:
from wildspeech_m84.ledger import preregister,Ledger
P=preregister(ROOT)
print("Plan hash:",P["plan_hash"])
display(pd.DataFrame(P["comparisons"]))
print("Ledger:",ROOT/"ledger"/"events.jsonl")

## 6. Execute the complete Phase 0 reproducible pipeline

This performs all 100 reference folds, saves each fold checkpoint, verifies OOF completeness, computes both R² estimands, writes the 14-feature registry, tests independent singleton test inference, fits the full reference on training rows only, and generates the **submission file** if parity passes. Cached runs can be safely resumed/reused by immutable hashes.

The code never claims an online leaderboard result.

In [ ]:
started=time.perf_counter()
RESULT=run_phase0(
    drive_root=DRIVE,
    data_root_override=DATA_ROOT_OVERRIDE,
    output_root_override=OUTPUT_ROOT_OVERRIDE,
    m8_root_override=M8_ROOT_OVERRIDE,
    m7_profile_override=M7_PROFILE_OVERRIDE,
    run_legacy_research=RUN_LEGACY_RESEARCH_ONLY,
    repository_root=CHECKOUT,
    require_parity=REQUIRE_REFERENCE_PARITY,
)
print("Phase 0 execution or verified cache reuse completed in",round(time.perf_counter()-started,1),"seconds")
print("Reference parity:",RESULT["parity_pass"],"| Gate:",RESULT["phase0_gate"])
display(Markdown((ROOT/"parity"/"M84_BASELINE_PARITY.md").read_text()))

## 7. Reference R² results: mean of folds and pooled OOF

Published reference comparison requires *mean of 50 fold R²*; the primary competition-oriented summary uses *10-repeat mean pooled OOF R²*. Do not conflate these estimands.

In [ ]:
reference_id=RESULT["reference_run_id"]
RUN_DIR=ROOT/"runs"/reference_id
FOLD_METRICS=pd.read_parquet(RUN_DIR/"fold_metrics.parquet")
POOLED=pd.read_parquet(RUN_DIR/"pooled_metrics.parquet")
# Backfill for older completed M8.4 runs: derive subgroup metrics from cached
# held-out predictions. This does NOT refit any model or alter the run ledger.
OOF=pd.read_parquet(RUN_DIR/"oof_predictions_PRIVATE.parquet")
instrument_path=RUN_DIR/"instrument_metrics.parquet"
if not instrument_path.is_file():
    from wildspeech_m84.metrics import summarize_oof
    _,_,recovered=summarize_oof(OOF,FOLDS,D.ids,Y,instruments=D.instruments)
    if recovered.empty:
        raise ValueError("Unable to reconstruct instrument metrics from validated OOF")
    temporary=instrument_path.with_name("instrument_metrics.tmp.parquet")
    recovered.to_parquet(temporary,index=False)
    temporary.replace(instrument_path)
    print("Repaired missing instrument metrics from cached OOF:",instrument_path)
INSTRUMENT=pd.read_parquet(instrument_path)
print("Reference mean fold R²:",FOLD_METRICS[FOLD_METRICS.scheme=="student"].r2.mean())
print("Reference mean pooled OOF R²:",POOLED[POOLED.scheme=="student"].r2.mean())
print("School-disjoint mean pooled OOF R²:",POOLED[POOLED.scheme=="school"].r2.mean())
display(POOLED.groupby("scheme")[["r2","mae","rmse"]].agg(["mean","std"]).round(5))
print("Private OOF:",RUN_DIR/"oof_predictions_PRIVATE.parquet")

## 8. Figure: 50 locked-fold R² scores and between-repeat variability

In [ ]:
FIGS=ROOT/"figures";FIGS.mkdir(exist_ok=True,parents=True)
f=FOLD_METRICS[FOLD_METRICS.scheme=="student"].sort_values(["repeat","fold"])
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(range(len(f)),f.r2,".-",ms=5,lw=1)
ax.axhline(.193,color="black",ls="--",label="Published mean-fold R² 0.193")
ax.axhline(f.r2.mean(),color="teal",ls=":",label="M8.4 mean-fold R²")
ax.set(xlabel="Outer fold index (5×10)",ylabel="Held-out fold R²",
       title="Reference RidgeCV: all 50 fixed student folds")
ax.legend();fig.tight_layout()
fig.savefig(FIGS/"01_reference_50fold_r2.png",dpi=170);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4))
for label,g in POOLED.groupby("scheme"):
    ax.plot(g.repeat,g.r2,"o-",label=label)
ax.set(xlabel="Repeat",ylabel="Pooled OOF R²",title="Pooled R² across 10 repeats")
ax.legend();fig.tight_layout()
fig.savefig(FIGS/"02_pooled_r2_repeats.png",dpi=170);plt.show();plt.close(fig)

## 9. Figure: instrument-specific generalization

Only report a subgroup R² when it has adequate training-sample support and nonzero target variance. These are **held-out training outcomes**, not hidden-test labels.

In [ ]:
display(INSTRUMENT.groupby(["scheme","instrument"])[["r2","n","sse"]].mean().round(4))
instrument_student=INSTRUMENT[INSTRUMENT.scheme=="student"]
fig,ax=plt.subplots(figsize=(8,4))
for name,g in instrument_student.groupby("instrument"):
    ax.plot(g.repeat,g.r2,marker="o",label=name)
ax.set(xlabel="Repeat",ylabel="Instrument pooled OOF R²",
       title="Held-out scores by instrument")
ax.legend();fig.tight_layout()
fig.savefig(FIGS/"03_instrument_robustness.png",dpi=170);plt.show();plt.close(fig)

## 10. OOF calibration and target error atlas

Use one prediction per student (repeat 0), not all repeated OOF predictions as independent records.

In [ ]:
d=OOF[(OOF.scheme=="student")&(OOF.repeat==0)].sort_values("pos")
assert len(d)==len(Y)
fig,axs=plt.subplots(1,2,figsize=(12,5))
axs[0].scatter(d.y_true,d.y_pred,s=13,alpha=.5)
lo=min(d.y_true.min(),d.y_pred.min());hi=max(d.y_true.max(),d.y_pred.max())
axs[0].plot([lo,hi],[lo,hi],"k--")
axs[0].set(xlabel="True reading score",ylabel="OOF prediction",
           title="Reference calibration (held-out)")
axs[1].scatter(d.y_pred,d.y_true-d.y_pred,s=13,alpha=.5)
axs[1].axhline(0,color="black",ls="--")
axs[1].set(xlabel="OOF predicted score",ylabel="True − prediction",
           title="Residual diagnostics")
fig.tight_layout();fig.savefig(FIGS/"04_calibration_residuals.png",dpi=170);plt.show();plt.close(fig)

## 11. Audit provenance registry

A model feature can be reference-protocol metadata **and** an acoustic nuisance proxy simultaneously. The registry records separate source and modeling roles. Test coverage is audit-only and never determines fitted parameters.

In [ ]:
REGISTRY=pd.read_csv(ROOT/"features"/"registry.csv")
display(REGISTRY[["feature_name","family","source_file","uses_item_score","uses_audio",
                  "nuisance","is_duplicate","train_coverage","test_coverage_audit_only"]])
assert len(REGISTRY)==14
assert not REGISTRY.allowed_in_speech_arm.fillna(False).any()
print("Feature names match official reference exactly:",REGISTRY.feature_name.tolist()==REFERENCE_COLUMNS)

## 12. Validate submission artifact independently

A candidate CSV is generated only after exact reference parity passes. Check row count/order, absence of NaNs, SHA-256, and proof that one student's prediction does not depend on other test rows. This cell does **not** upload anything to the competition.

In [ ]:
from wildspeech_m84.data import digest_bytes
if RESULT["parity_pass"]:
    sub=ROOT/"submissions"/"reference_submission.csv"
    frame=pd.read_csv(sub,dtype={"subject_id":str})
    assert list(frame)==["subject_id","outcome_score"]
    assert frame.subject_id.tolist()==D.submission_format.index.tolist()
    assert frame.outcome_score.notna().all()
    assert np.isfinite(frame.outcome_score.to_numpy()).all()
    assert digest_bytes(sub)==(ROOT/"submissions"/"reference_submission.sha256").read_text().strip()
    print("Submission ready (NOT submitted):",sub)
    print("SHA-256:",digest_bytes(sub))
else:
    print("NO-GO: reference parity not yet reproduced; do NOT submit.")

## 13. Historical-model parity is separate, never silently substituted

M7.3's original global source is not in GitHub main, so it is marked NOT_REPRODUCED. Optional M8.3 reruns use the fixed folds for research, but the legacy feature bank must pass a fresh train-only preprocessing audit before entering the official promotion gate. Never interpret the historical without_audio family as genuinely speech-free.

In [ ]:
print("Historical status:")
display(pd.DataFrame([{"model":k,"status":v} for k,v in RESULT["historical_status"].items()]))
if RESULT["comparisons"]:
    display(pd.DataFrame(RESULT["comparisons"])[
        ["baseline","challenger","mean_fold_delta_r2","corrected_t",
         "win_fraction","school_pooled_delta_r2","gate"]])
else:
    print("No invented old R² scores. Set RUN_LEGACY_RESEARCH_ONLY=True only for labeled diagnostics.")

## 14. Experiment ledger and invariants

Every run must record the Git SHA, feature hash, fold hash, status, predictions path and metrics. Outputs with mismatched hashes must never enter paired model comparisons.

In [ ]:
ledger=pd.read_parquet(ROOT/"ledger"/"experiments.parquet")
cols=[c for c in ["run_id","model_id","status","git_sha","feature_set_hash","fold_hash",
                  "mean_fold_r2","mean_pooled_oof_r2","oof_path"] if c in ledger]
display(ledger[cols])
assert (ledger[ledger.model_id=="REF_RIDGE14"].status=="COMPLETE").all()
assert (ledger[ledger.model_id=="REF_RIDGE14"].fold_hash==FOLD_HASH).all()
print("Ledger events:",ROOT/"ledger"/"events.jsonl")

## 15. Phase 0 gate, Phase 1 handoff and competition submission

**GO criteria:** exact reference mean-of-fold R² in [0.188,0.198], immutable 100 folds, complete and reproducible OOF, 14/14 known feature lineage, independent test inference, strict CSV schema and checksum.

**CONDITIONAL GO:** reference passes but exact historical adapter unavailable; start Phase 1 against the frozen reference only. **NO GO:** baseline mismatch, any fold leakage, test-row fitting, failed inference invariance or changing fold hash.

**Next M0+ predeclared experiments:** (A) stronger regularized reference stack; (B) bundled S2–S5 item-level, theta splines, instrument interactions and missing task signals; (C) block-by-block ablations. All item difficulty statistics must be learned inside training folds.

If approved for an external leaderboard check: upload only the validated CSV and record the *actual* receipt and public score. **Do not tune to the public leaderboard.**